# Stage 0 Simple LoRA Baseline (A100)

Thin Colab orchestration notebook for a simple LoRA baseline with `f03_multimodal_letter`, an optional eval-only rerun cell with a larger validation batch size, and a summary cell for the repo's existing results tracking.


## 1. Mount Google Drive

Mount Google Drive before running the same Colab bootstrap flow used in `starter_notebook.ipynb`.


In [1]:
from google.colab import drive

drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Configure Paths

Set the repo checkout path, Drive output directory, and optional data override here. The setup cell below uses the same upload-widget bootstrap code as `starter_notebook.ipynb`.

If the upload widget appears, select your local `.env` file and rerun the setup cell.


In [2]:
from pathlib import Path

REPO_URL = "https://github.com/Demetri65/dl-kaggle-competition-final.git"
REPO_REF = "main"
REPO_DIR = Path("/content/dl-kaggle-competition-final")  # Change this if you want the repo checkout elsewhere
SOURCE_ENV = Path("/content/.env")
UPLOADER_KEY = "_env_uploader"
DRIVE_OUTPUT_DIR = "/content/drive/MyDrive/p2p_runs/notebook_c_a100"
DATA_DIR_OVERRIDE = ""  # Optional: set this to an existing dataset directory for the run commands
EVAL_BATCH_SIZE = 64  # Number of validation examples handed to the evaluator per outer batch
INFERENCE_COMPLETION_BATCH_SIZE = 64  # Number of flattened answer completions scored per model forward pass
TRAINED_F03_ARTIFACT_DIR = "/content/drive/MyDrive/p2p_runs/notebook_c_a100/full_f03/f03_multimodal_letter_seed42/model"
ARTIFACT_SEARCH_ROOT = "/content/drive/MyDrive"  # Search the full Drive by default when auto-discovering the saved f03 artifact dir


## 3. Prepare The Repo

This is the same Colab environment setup flow used in `starter_notebook.ipynb`: upload `.env`, sync the repo, then run `scripts/bootstrap_colab.sh`.


In [3]:
# ── 0. Colab setup ───────────────────────────────────────────────
import subprocess
import ipywidgets as widgets
from IPython.display import display

def get_uploaded_file(uploader):
    value = uploader.value

    if isinstance(value, dict):
        filename, uploaded_file = next(iter(value.items()))
        if isinstance(uploaded_file, dict):
            content = uploaded_file.get("content", uploaded_file.get("data"))
        else:
            content = uploaded_file
    else:
        uploaded_file = value[0]
        if isinstance(uploaded_file, dict):
            filename = uploaded_file["name"]
            content = uploaded_file["content"]
        else:
            filename = uploaded_file.name
            content = uploaded_file.content

    payload = content.tobytes() if hasattr(content, "tobytes") else bytes(content)
    return filename, payload

ready_to_bootstrap = SOURCE_ENV.exists()

if ready_to_bootstrap:
    print(f"Using existing {SOURCE_ENV}")
else:
    uploader = globals().get(UPLOADER_KEY)
    if uploader is None:
        uploader = widgets.FileUpload(accept=".env", multiple=False, description="Upload .env")
        globals()[UPLOADER_KEY] = uploader

    if not uploader.value:
        display(uploader)
        print("Select your local .env file in the upload widget above, then rerun this cell.")
    else:
        filename, payload = get_uploaded_file(uploader)
        SOURCE_ENV.write_bytes(payload)
        SOURCE_ENV.chmod(0o600)
        print(f"Saved {filename} to {SOURCE_ENV}")
        uploader.close()
        globals().pop(UPLOADER_KEY, None)
        ready_to_bootstrap = True

if ready_to_bootstrap:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", REPO_REF, REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--depth", "1", "origin", REPO_REF], check=True)
        subprocess.run(["git", "-C", str(REPO_DIR), "reset", "--hard", "FETCH_HEAD"], check=True)

    print(f"Repo synced to latest origin/{REPO_REF} at {REPO_DIR}")
    result = subprocess.run(
        ["bash", "scripts/bootstrap_colab.sh"],
        cwd=REPO_DIR,
        capture_output=True,
        text=True,
    )
    if result.stdout:
        print(result.stdout, end="")
    if result.returncode != 0:
        if result.stderr:
            print(result.stderr, end="")
        raise RuntimeError(f"scripts/bootstrap_colab.sh failed with exit code {result.returncode}")


Using existing /content/.env
Repo synced to latest origin/main at /content/dl-kaggle-competition-final
Using Kaggle CLI Kaggle CLI 2.0.1
pixels-to-predictions.zip: Skipping, found more recently modified local copy (use --force to force download)
Extracting pixels-to-predictions.zip
Competition files downloaded to: /content/dl-kaggle-competition-final/data


## 4. Helpers

These helpers keep the runnable cells thin and route jobs through the existing repo scripts.


In [4]:
import os
import shlex
import subprocess
from pathlib import Path

REPO_ROOT = REPO_DIR.resolve()
Path(DRIVE_OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
os.chdir(REPO_ROOT)

COMMON_OVERRIDES = []
if DATA_DIR_OVERRIDE:
    COMMON_OVERRIDES.append(f"data.data_dir={DATA_DIR_OVERRIDE}")

def with_common_overrides(overrides):
    return [*COMMON_OVERRIDES, *overrides]

def extend_with_overrides(args, overrides):
    for override in overrides:
        args.extend(["--set", override])

def run_repo_command(args):
    command = ["python3", *args]
    print("$", " ".join(shlex.quote(part) for part in command))
    result = subprocess.run(command, cwd=REPO_ROOT, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout, end="")
    if result.stderr:
        print(result.stderr, end="")
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}")
    return result

def resolve_saved_artifact_dir(explicit_dir: str, search_root: str, experiment_hint: str = "f03") -> Path:
    if explicit_dir:
        artifact_dir = Path(explicit_dir)
        if artifact_dir.name == "model":
            artifact_dir = artifact_dir.parent
        if artifact_dir.exists() and (artifact_dir / "model").exists() and (artifact_dir / "processor").exists():
            return artifact_dir
        print(f"Explicit artifact dir was not usable, falling back to auto-discovery: {artifact_dir}")

    root = Path(search_root)
    if not root.exists():
        raise FileNotFoundError(f"Artifact search root not found: {root}")

    candidates = []
    for marker_name in ("adapter_model.safetensors", "model_adapters.safetensors", "adapter_config.json"):
        for marker_path in root.rglob(marker_name):
            if marker_path.parent.name != "model":
                continue
            run_dir = marker_path.parent.parent
            if (run_dir / "model").exists() and (run_dir / "processor").exists():
                candidates.append(run_dir)

    unique_candidates = sorted({candidate.resolve() for candidate in candidates})
    hinted_candidates = [candidate for candidate in unique_candidates if experiment_hint in str(candidate).lower()]
    if len(hinted_candidates) == 1:
        return hinted_candidates[0]
    if not hinted_candidates and len(unique_candidates) == 1:
        return unique_candidates[0]

    candidate_list = hinted_candidates or unique_candidates
    if len(candidate_list) > 1:
        candidate_list = sorted(
            candidate_list,
            key=lambda candidate: max(path.stat().st_mtime for path in candidate.rglob('*') if path.exists()),
            reverse=True,
        )
        print(f"Auto-discovery found multiple candidates; using the most recently updated one: {candidate_list[0]}")
        return candidate_list[0]

    formatted = "\n".join(str(candidate) for candidate in candidate_list[:20]) or "<none found>"
    raise RuntimeError(
        "Could not resolve a saved artifact dir automatically. "
        f"Set TRAINED_F03_ARTIFACT_DIR explicitly. Candidates:\n{formatted}"
    )

F03_ARTIFACT_DIR = resolve_saved_artifact_dir(TRAINED_F03_ARTIFACT_DIR, ARTIFACT_SEARCH_ROOT, experiment_hint="f03")

print(f"Repo root: {REPO_ROOT}")
print(f"Drive output dir: {DRIVE_OUTPUT_DIR}")
print(f"Resolved f03 artifact dir: {F03_ARTIFACT_DIR}")
if DATA_DIR_OVERRIDE:
    print(f"Data override: {DATA_DIR_OVERRIDE}")


Repo root: /content/dl-kaggle-competition-final
Drive output dir: /content/drive/MyDrive/p2p_runs/notebook_c_a100
Resolved f03 artifact dir: /content/drive/MyDrive/p2p_runs/notebook_c_a100/full_f03/f03_multimodal_letter_seed42


## 6. Eval-Only `f03` Rerun With Inference Batching

Reuse the trained `f03_multimodal_letter` artifact, rerun validation through the normal pipeline, and increase inference-time completion batching to better use the A100.


In [ ]:
eval_args = [
    "scripts/run_experiment.py",
    "--experiment", "f03_multimodal_letter",
    "--output-dir", f"{DRIVE_OUTPUT_DIR}/re_eval_f03",
]
extend_with_overrides(eval_args, with_common_overrides([
    "training.epochs=0",
    f"training.eval_batch_size={EVAL_BATCH_SIZE}",
    f"scoring.max_completion_batch_size={INFERENCE_COMPLETION_BATCH_SIZE}",
    f"runtime.eval_artifact_dir={F03_ARTIFACT_DIR}",
]))
run_repo_command(eval_args)

resolved_config_path = Path(DRIVE_OUTPUT_DIR) / "re_eval_f03" / "f03_multimodal_letter_seed42" / "resolved_config.yaml"
if resolved_config_path.exists():
    print(f"Resolved config written to: {resolved_config_path}")
    print("Verify batching with:")
    print(f"grep -n 'eval_batch_size\\|max_completion_batch_size' {resolved_config_path}")


## 8. Summarize Results

Print the top validation runs from `results/experiments.csv`.


In [ ]:
summary_args = [
    "scripts/summarize_results.py",
    "--sort-by", "val_accuracy",
    "--top", "20",
]
run_repo_command(summary_args)
